# MNIST CNN Training

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

Using: cuda


In [2]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.conv2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return self.fc(x)

    def get_activations(self, x):
        a1 = self.conv1(x)
        a2 = self.conv2(a1)
        return [a1, a2]

In [3]:
train_tf = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

train_data = datasets.MNIST('./data', train=True, download=True, transform=train_tf)
test_data  = datasets.MNIST('./data', train=False, transform=test_tf)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=1000, num_workers=2)

print(f'Train: {len(train_data)} | Test: {len(test_data)}')

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 352kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.27MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.3MB/s]

Train: 60000 | Test: 10000


In [4]:
model = CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (model(imgs).argmax(1) == labels).sum().item()

    print(f'Epoch {epoch+1}/10  Loss: {total_loss/len(train_loader):.4f}  Acc: {correct/len(test_data)*100:.2f}%')

Epoch 1/10  Loss: 0.4264  Acc: 97.91%
Epoch 2/10  Loss: 0.1494  Acc: 97.86%
Epoch 3/10  Loss: 0.1127  Acc: 98.84%
Epoch 4/10  Loss: 0.0913  Acc: 99.07%
Epoch 5/10  Loss: 0.0795  Acc: 99.18%
Epoch 6/10  Loss: 0.0749  Acc: 99.14%
Epoch 7/10  Loss: 0.0665  Acc: 98.96%
Epoch 8/10  Loss: 0.0634  Acc: 99.15%
Epoch 9/10  Loss: 0.0574  Acc: 99.16%
Epoch 10/10  Loss: 0.0577  Acc: 99.01%


In [5]:
torch.save(model.state_dict(), 'mnist_cnn.pth')

from google.colab import files
files.download('mnist_cnn.pth')
print('✅ Done! Drop mnist_cnn.pth into your project folder.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Drop mnist_cnn.pth into your project folder.
